In [1]:
import pandas as pd
import numpy as np
import pymc as pm
from pymc.model import Model
import os
import glob

In [2]:
def process_mrt_data(df):

  df = df.query('viability == 1')
  df = df.copy()
  df['decision_date'] = pd.to_datetime(df['decision_time']).dt.date

  df['user_start_day_dt'] = pd.to_datetime(df['user_start_day']).dt.date
  df['decision_date'] = pd.to_datetime(df['decision_date'])
  df['user_start_day_dt'] = pd.to_datetime(df['user_start_day'])

  df['state_day_type'] = pd.to_datetime(df['decision_time']).apply(lambda x: 1 if x.weekday() >= 5 else 0)

  df = df.drop(columns = ['state_modif'])

  df['state_day_in_study'] = (df['decision_date'] - df['user_start_day_dt']).dt.days + 1
  df['state_day_in_study'] = 0
  # df['state_app_engage'] = 0
  # (df['state_day_in_study'] - 35.5) / 34.5

  # desired_order = ['user_id', 'decision_time', 'action', 'quality', 'state_tod',
                #  'state_b_bar', 'state_a_bar', 'state_app_engage', 
                #  'state_bias']
  desired_order = ['user_id', 'decision_time', 'action', 'quality', 'state_tod',
                 'state_b_bar', 'state_a_bar', 'state_app_engage', 'state_day_type',
                 'state_bias']
  df = df[desired_order]

  return df


def get_user_data(df, user_id):
  return df[df['user_id'] == user_id]


In [3]:
script_dir = os.getcwd()
mrt_data_path = os.path.join(script_dir, '../../../OralyticsMRT/oralytics_mrt_data.csv')

MRT_DATA = pd.read_csv(mrt_data_path)
MRT_DATA = process_mrt_data(MRT_DATA)
MRT_USERS = MRT_DATA['user_id'].unique().tolist()

# print(MRT_DATA.tail())
# print(MRT_DATA['state_b_bar'].unique())
# print(MRT_DATA['state_a_bar'].unique())
# print(len(MRT_DATA['user_id'].unique()))
# print(len(MRT_DATA[MRT_DATA['user_id']=='robas+127@developers.pg.com']['decision_time'].unique()))

# save MRT_DATA


In [4]:
# calculate mean and variance of state features
state_features = ['state_tod', 'state_b_bar', 'state_a_bar', 'state_app_engage', 'state_day_type', 'state_bias']
for feature in state_features:
    print(f"{feature} mean: {MRT_DATA[feature].mean()}, variance: {MRT_DATA[feature].var(ddof=1)}")


state_tod mean: 0.49970214455917394, variance: 0.25002473502469247
state_b_bar mean: -0.27395626542757234, variance: 0.2723113390689649
state_a_bar mean: -0.3249013449101297, variance: 0.1296687316789945
state_app_engage mean: 0.4323868149324861, variance: 0.24545282702502463
state_day_type mean: 0.28594122319301035, variance: 0.20419911401093543
state_bias mean: 1.0, variance: 0.0


In [7]:
# discretize b_bar and a_bar
bins = np.quantile(MRT_DATA['state_b_bar'], [0, 0.2, 0.4, 0.6, 0.8])
MRT_DATA['state_b_bar'] = np.digitize(MRT_DATA['state_b_bar'], bins)-1
bins = np.quantile(MRT_DATA['state_a_bar'], [0, 0.2, 0.4, 0.6, 0.8])
MRT_DATA['state_a_bar'] = np.digitize(MRT_DATA['state_a_bar'], bins)-1

print(MRT_DATA['state_b_bar'].value_counts())
print(MRT_DATA['state_a_bar'].value_counts())

state_b_bar
2    2015
4    2015
0    2014
1    2014
3    2014
Name: count, dtype: int64
state_a_bar
4    2022
1    2016
2    2014
0    2013
3    2007
Name: count, dtype: int64


In [25]:
# save MRT_DATA
# Save MRT_DATA to CSV
mrt_data_save_path = os.path.join(script_dir, "../../sim_env_data/oralytics_mrt_data_processed.csv")
MRT_DATA.to_csv(mrt_data_save_path, index=False)
print(f"Saved processed MRT_DATA to {mrt_data_save_path}")


Saved processed MRT_DATA to /Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/budgeted_oral/src/dev_scripts/../../sim_env_data/oralytics_mrt_data_processed.csv


In [26]:
def merge_message_types(mrt_data, message_type_dir):
    """
    Merge message_type data from individual user CSV files into MRT_DATA.
    
    Args:
        mrt_data: DataFrame with MRT data
        message_type_dir: Directory containing user-specific CSV files
    
    Returns:
        DataFrame with message_type column added
    """
    
    # Get all CSV files in the message type directory
    csv_files = glob.glob(os.path.join(message_type_dir, "*.csv"))
    
    # List to store all message type data
    all_message_data = []
    
    print(f"Found {len(csv_files)} message type files")
    
    for file_path in csv_files:
        # Extract user_id from filename (remove .csv extension)
        user_id = os.path.basename(file_path).replace('.csv', '')
        

        # Read the user's message type data
        user_message_data = pd.read_csv(file_path)
        
        # Add user_id column
        user_message_data['user_id'] = user_id
        
        # Select only the columns we need for merging
        merge_cols = ['user_id', 'decision_time', 'message_type']
        user_message_data = user_message_data[merge_cols]
        
        all_message_data.append(user_message_data)
            

    
    # Combine all message type data

    message_type_df = pd.concat(all_message_data, ignore_index=True)
    
    # Convert decision_time to datetime for both DataFrames
    mrt_data_copy = mrt_data.copy()
    mrt_data_copy['decision_time'] = pd.to_datetime(mrt_data_copy['decision_time'])
    message_type_df['decision_time'] = pd.to_datetime(message_type_df['decision_time'])
    
    # Merge on user_id and decision_time
    merged_data = mrt_data_copy.merge(
        message_type_df, 
        on=['user_id', 'decision_time'], 
        how='left'
    )
    
    print(f"Original MRT_DATA shape: {mrt_data.shape}")
    print(f"Message type data shape: {message_type_df.shape}")
    print(f"Merged data shape: {merged_data.shape}")
    print(f"Rows with message_type: {merged_data['message_type'].notna().sum()}")
    
    return merged_data



In [27]:
# Merge message type data with MRT_DATA
message_type_path = os.path.join(script_dir, '../../../OralyticsMRT/MRT_messagetype_parsing/message_type_df/')

# Apply the merge function
MRT_DATA_WITH_MESSAGE_TYPE = merge_message_types(MRT_DATA, message_type_path)


Found 72 message type files
Original MRT_DATA shape: (10072, 10)
Message type data shape: (10224, 3)
Merged data shape: (10072, 11)
Rows with message_type: 3382


In [28]:
# Create simplified message type categories
MRT_DATA_WITH_MESSAGE_TYPE['message_type_category'] = MRT_DATA_WITH_MESSAGE_TYPE['message_type'].str.replace(r'-\d{2}$', '', regex=True)

# Check the distribution
print("Message type category distribution:")
print(MRT_DATA_WITH_MESSAGE_TYPE['message_type_category'].value_counts(dropna=False))
print(MRT_DATA_WITH_MESSAGE_TYPE.head())

Message type category distribution:
message_type_category
NaN    6690
RP     1166
SR     1142
FB      605
QA      469
Name: count, dtype: int64
                       user_id       decision_time  action  quality  \
0  robas+119@developers.pg.com 2023-09-22 05:00:00     0.0        0   
1  robas+126@developers.pg.com 2023-09-22 06:45:00     1.0       54   
2  robas+119@developers.pg.com 2023-09-22 18:00:00     1.0        0   
3  robas+119@developers.pg.com 2023-09-23 08:00:00     0.0      129   
4  robas+126@developers.pg.com 2023-09-22 20:15:00     1.0        0   

   state_tod  state_b_bar  state_a_bar  state_app_engage  state_day_type  \
0        0.0            0            0               1.0               0   
1        0.0            0            0               1.0               0   
2        1.0            0            0               1.0               0   
3        0.0            0            0               1.0               1   
4        1.0            0            0           

In [29]:
def get_batch_data(df, user_id):
  user_df = get_user_data(df, user_id)
  states_df = user_df.filter(regex='state_*')
  outcomes = user_df['quality']
  actions = user_df['message_type_category']

  return np.array(states_df), np.array(outcomes), np.array(actions)

In [30]:
# Check for rows with action = 1.0 but message_type is NaN
action_1_no_message = MRT_DATA_WITH_MESSAGE_TYPE[(MRT_DATA_WITH_MESSAGE_TYPE['action'] == 1.0) & (MRT_DATA_WITH_MESSAGE_TYPE['message_type'].isna())]

print(f"Rows with action=1.0 but message_type is NaN: {len(action_1_no_message)}")
print(f"Total rows with action=1.0: {len(MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['action'] == 1.0])}")
print(f"Percentage of action=1.0 rows missing message_type: {len(action_1_no_message) / len(MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['action'] == 1.0]) * 100:.1f}%")

if len(action_1_no_message) > 0:
    print(f"\nFirst few examples:")
    print(action_1_no_message[['user_id', 'decision_time', 'action', 'message_type']].head())


Rows with action=1.0 but message_type is NaN: 336
Total rows with action=1.0: 3438
Percentage of action=1.0 rows missing message_type: 9.8%

First few examples:
                         user_id       decision_time  action message_type
133  robas+135@developers.pg.com 2023-10-07 20:00:00     1.0          NaN
252  robas+126@developers.pg.com 2023-10-20 06:45:00     1.0          NaN
367  robas+126@developers.pg.com 2023-10-27 20:15:00     1.0          NaN
368  robas+126@developers.pg.com 2023-10-28 08:00:00     1.0          NaN
371  robas+199@developers.pg.com 2023-10-27 22:00:00     1.0          NaN


## Fitting Models
---

### Helpers

In [31]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

def build_zip_model(X, A, Y):
  model = pm.Model()
  with Model() as model:
    d = X.shape[1]
    
    # Baseline parameters (when A is NaN)
    w_b = pm.MvNormal('w_b', mu=np.zeros(d), cov=np.eye(d), shape=d)
    w_p = pm.MvNormal('w_p', mu=np.zeros(d), cov=np.eye(d), shape=d)
    
    # Categorical effects for each message type (excluding NaN baseline)
    # Get unique categories excluding NaN
    A_series = pd.Series(A)
    categories = [cat for cat in A_series.unique() if pd.notna(cat)]
    n_categories = len(categories)
    
    # Effects for each category
    delta_b = pm.MvNormal('delta_b', mu=np.zeros(d), cov=np.eye(d), shape=(n_categories, d))
    delta_p = pm.MvNormal('delta_p', mu=np.zeros(d), cov=np.eye(d), shape=(n_categories, d))
    
    # Create design matrix for categorical effects
    bern_term = X @ w_b
    poisson_term = X @ w_p
    
    # Add categorical effects
    for i, category in enumerate(categories):
        mask = (A == category)
        if mask.any():
            bern_term = bern_term + mask * (X @ delta_b[i])
            poisson_term = poisson_term + mask * (X @ delta_p[i])
    
    # R = pm.ZeroInflatedPoisson("likelihood", psi=1 - sigmoid(bern_term), mu=np.exp(poisson_term), observed=Y)
    R = pm.Normal("likelihood", mu=bern_term, sigma=1, observed=Y)

  return model

def run_zip_map_for_users(users_states, users_actions, users_rewards, num_restarts):
  model_params = {}

  for user_id in users_states.keys():
    print("FOR USER: ", user_id)
    user_states = users_states[user_id]
    d = user_states.shape[1]
    user_actions = users_actions[user_id]
    user_rewards = users_rewards[user_id]
    
    # Get number of categories (excluding NaN)
    # Convert to pandas Series to handle mixed types properly
    user_actions_series = pd.Series(user_actions)
    categories = [cat for cat in user_actions_series.unique() if pd.notna(cat)]
    n_categories = len(categories)
    
    logp_vals = np.empty(shape=(num_restarts,))
    # Calculate total parameter size: w_b(d) + delta_b(n_cat*d) + w_p(d) + delta_p(n_cat*d)
    total_params = d + n_categories * d
    #  + d + n_categories * d
    param_vals = np.empty(shape=(num_restarts, total_params))
    
    for seed in range(num_restarts):
      model = build_zip_model(user_states, user_actions, user_rewards)
      np.random.seed(seed)
      
      # Initialize parameters with correct shapes
      init_params = {
        'w_b': np.random.randn(d), 
        'delta_b': np.random.randn(n_categories, d)
        # 'w_p': np.random.randn(d), 
        # 'delta_p': np.random.randn(n_categories, d)
      }
      
      with model:
        map_estimate = pm.find_MAP(start=init_params)

      w_b = map_estimate['w_b']
      delta_b = map_estimate['delta_b']
      # w_p = map_estimate['w_p']
      # delta_p = map_estimate['delta_p']
      logp_vals[seed] = model.compile_logp()(map_estimate)
      param_vals[seed] = np.concatenate((w_b, delta_b.flatten()), axis=None)
    model_params[user_id] = param_vals[np.argmax(logp_vals)]

  return model_params

### Execution

In [32]:
users_states = {}
users_rewards = {}
users_actions = {}
for user_id in MRT_USERS:
    states, rewards, actions = get_batch_data(MRT_DATA_WITH_MESSAGE_TYPE, user_id)
    users_rewards[user_id] = np.log(rewards + 0.1)
    users_actions[user_id] = actions
    users_states[user_id] = states

In [33]:
zip_model_params = run_zip_map_for_users(users_states, users_actions, users_rewards, num_restarts=5)

FOR USER:  robas+119@developers.pg.com


Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

Output()

## Saving Parameter Values
---

In [34]:
def create_model_columns(feature_names, message_categories):
    """
    Create column names for the categorical ZIP model parameters.
    
    Args:
        feature_names: List of feature names (e.g., ['state_tod', 'state_b_bar', ...])
        message_categories: List of message type categories (e.g., ['QA', 'SR', ...])
    
    Returns:
        List of column names
    """
    columns = ['User']
    
    # Baseline parameters (Bernoulli and Poisson)
    for feature in feature_names:
        columns.append(f'{feature}.Base')
    # for feature in feature_names:
        # columns.append(f'{feature}.Base.Poisson')
    
    # Category-specific parameters for each message type
    for category in message_categories:
        for feature in feature_names:
            columns.append(f'{feature}.{category}')
        # for feature in feature_names:
            # columns.append(f'{feature}.{category}.Poisson')
    
    return columns

# Get unique message categories from the data (excluding NaN)
message_categories = [cat for cat in MRT_DATA_WITH_MESSAGE_TYPE['message_type_category'].unique() if pd.notna(cat)]
feature_names = ['state_tod', 'state_b_bar.norm', 'state_a_bar.norm', 'state_app_engage', 'state_day_type', 'state_bias']

# Create the column names dynamically
non_stat_zip_model_columns = create_model_columns(feature_names, message_categories)

print(f"Message categories found: {message_categories}")
print(f"Number of columns: {len(non_stat_zip_model_columns)}")
print("First 10 columns:", non_stat_zip_model_columns[:10])


Message categories found: ['RP', 'SR', 'QA', 'FB']
Number of columns: 31
First 10 columns: ['User', 'state_tod.Base', 'state_b_bar.norm.Base', 'state_a_bar.norm.Base', 'state_app_engage.Base', 'state_day_type.Base', 'state_bias.Base', 'state_tod.RP', 'state_b_bar.norm.RP', 'state_a_bar.norm.RP']


In [35]:
def fill_missing_parameters_with_averages(zip_model_params, non_stat_zip_model_columns, feature_names, message_categories):
    """
    Correctly fill missing parameters by mapping them to the right columns
    based on the user's actual message type categories.
    """
    import numpy as np
    
    rows = []
    
    for user in zip_model_params.keys():
        values = zip_model_params[user]
        new_row = {'User': user}
        
        # Get this user's actual message type categories
        user_data = MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['user_id'] == user]
        user_categories = [cat for cat in user_data['message_type_category'].unique() if pd.notna(cat)]
        
        # Map parameters to columns correctly
        param_idx = 0
        
        # 1. Baseline parameters (w_b and w_p)
        for feature in feature_names:
            if param_idx < len(values):
                new_row[f'{feature}.Base'] = values[param_idx]
                param_idx += 1
            else:
                new_row[f'{feature}.Base'] = np.nan
                
        # for feature in feature_names:
        #     if param_idx < len(values):
        #         new_row[f'{feature}.Base.Poisson'] = values[param_idx]
        #         param_idx += 1
        #     else:
        #         new_row[f'{feature}.Base.Poisson'] = np.nan
        
        # 2. Category-specific parameters (delta_b and delta_p)
        for category in message_categories:
            if category in user_categories:
                # User has this category, use their parameters
                for feature in feature_names:
                    if param_idx < len(values):
                        new_row[f'{feature}.{category}'] = values[param_idx]
                        param_idx += 1
                    else:
                        new_row[f'{feature}.{category}'] = np.nan
                        
                # for feature in feature_names:
                #     if param_idx < len(values):
                #         new_row[f'{feature}.{category}.Poisson'] = values[param_idx]
                #         param_idx += 1
                #     else:
                #         new_row[f'{feature}.{category}.Poisson'] = np.nan
            else:
                # User doesn't have this category, will fill with average later
                for feature in feature_names:
                    new_row[f'{feature}.{category}'] = np.nan
                # for feature in feature_names:
                #     new_row[f'{feature}.{category}.Poisson'] = np.nan
        
        rows.append(new_row)
    
    df = pd.DataFrame(rows, columns=non_stat_zip_model_columns)
    
    # Fill NaN values with column averages
    print("Filling missing parameters with averages...")
    for col in non_stat_zip_model_columns[1:]:  # Skip 'User' column
        if df[col].isna().any():
            avg_value = df[col].mean()
            num_missing = df[col].isna().sum()
            df[col] = df[col].fillna(avg_value)
            print(f"  {col}: filled {num_missing} missing values with average {avg_value:.4f}")
    
    return df

non_stat_zip_df = fill_missing_parameters_with_averages(zip_model_params, non_stat_zip_model_columns, feature_names, message_categories)

Filling missing parameters with averages...
  state_tod.QA: filled 1 missing values with average -0.0358
  state_b_bar.norm.QA: filled 1 missing values with average 0.0103
  state_a_bar.norm.QA: filled 1 missing values with average -0.0958
  state_app_engage.QA: filled 1 missing values with average -0.1129
  state_day_type.QA: filled 1 missing values with average 0.1786
  state_bias.QA: filled 1 missing values with average -0.1112


In [36]:
# # Check which users have fewer than 4 message type categories
# user_category_counts = {}

# for user_id in MRT_DATA_WITH_MESSAGE_TYPE['user_id'].unique():
#     user_data = MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['user_id'] == user_id]
#     user_categories = [cat for cat in user_data['message_type_category'].unique() if pd.notna(cat)]
#     user_category_counts[user_id] = len(user_categories)

# # Convert to DataFrame for easier analysis
# category_analysis = pd.DataFrame([
#     {'user_id': user, 'num_categories': count, 'categories': [cat for cat in MRT_DATA_WITH_MESSAGE_TYPE[MRT_DATA_WITH_MESSAGE_TYPE['user_id'] == user]['message_type_category'].unique() if pd.notna(cat)]}
#     for user, count in user_category_counts.items()
# ])

# print("Message type category distribution:")
# print(category_analysis['num_categories'].value_counts().sort_index())

# print(f"\nUsers with fewer than 4 categories:")
# users_less_than_4 = category_analysis[category_analysis['num_categories'] < 4]
# print(f"Count: {len(users_less_than_4)}")

# if len(users_less_than_4) > 0:
#     print("\nDetails:")
#     for _, row in users_less_than_4.iterrows():
#         print(f"  {row['user_id']}: {row['num_categories']} categories - {row['categories']}")

# print(f"\nUsers with exactly 4 categories:")
# users_with_4 = category_analysis[category_analysis['num_categories'] == 4]
# print(f"Count: {len(users_with_4)}")

# print(f"\nUsers with more than 4 categories:")
# users_more_than_4 = category_analysis[category_analysis['num_categories'] > 4]
# print(f"Count: {len(users_more_than_4)}")


In [37]:
non_stat_zip_df

,User,state_tod.Base,state_b_bar.norm.Base,state_a_bar.norm.Base,state_app_engage.Base,state_day_type.Base,state_bias.Base,state_tod.RP,state_b_bar.norm.RP,state_a_bar.norm.RP,...,state_a_bar.norm.QA,state_app_engage.QA,state_day_type.QA,state_bias.QA,state_tod.FB,state_b_bar.norm.FB,state_a_bar.norm.FB,state_app_engage.FB,state_day_type.FB,state_bias.FB
0,robas+119@developers.pg.com,-1.509843,0.744257,0.330155,-0.145052,-0.310714,-0.661583,3.848712,-1.384573,0.953109,...,0.688902,0.185498,0.928999,-0.626414,-0.689784,1.285081,-0.036247,1.659869,-1.805441,-0.693090
1,robas+126@developers.pg.com,-3.194775,0.584272,-0.023363,0.234801,0.114464,1.987550,2.709215,0.276352,-0.865030,...,-0.074568,0.638716,-0.888004,1.368101,-0.553383,-1.040952,0.349849,0.447481,-0.861371,-0.554136
2,digitaldentalcoach+214@gmail.com,0.371514,0.321480,-0.223193,-0.435772,-0.562303,2.739737,-0.000163,-0.162089,-0.261320,...,0.357351,-1.094466,-0.638708,1.254271,0.515101,0.400873,-0.342619,-1.298476,-2.170453,0.516667
3,robas+135@developers.pg.com,-1.089275,-0.335052,0.084702,-0.175729,0.379246,4.256320,0.126956,0.898003,-1.813087,...,-0.076761,1.547763,-1.897609,0.701442,0.000904,-0.290893,0.145849,0.476386,-2.272976,0.987677
4,robas+199@developers.pg.com,-3.707212,1.793727,-0.261431,-0.585413,0.191944,0.987376,0.006494,-0.859194,-0.002369,...,0.488688,-0.160284,0.760946,1.871881,-0.000564,1.392890,-0.399478,-0.664885,-0.600449,0.236704
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,digitaldentalcoach+249@gmail.com,-0.999418,-0.649605,-0.349544,0.539585,-0.316606,3.705164,-0.525233,1.735041,-0.588132,...,0.185796,-0.121084,2.131564,-0.318205,0.766516,0.664710,-0.289826,-0.894087,-0.169238,0.744448
68,digitaldentalcoach+271@gmail.com,-1.356434,0.202030,0.289114,0.209141,-0.666329,2.354569,-1.668773,-0.922458,0.046333,...,-0.654794,-0.263103,-0.135392,-0.643438,0.001591,-1.391805,0.529024,-0.814815,0.001075,1.383450
69,digitaldentalcoach+236@gmail.com,-2.989245,0.603518,0.314790,-0.206522,-0.492147,1.877159,-1.133608,-0.065931,-0.695777,...,-0.877321,-0.035511,2.854295,-0.305121,-0.000094,0.215208,0.116331,-0.174641,0.246479,0.071657
70,robas+118@developers.pg.com,-1.321516,0.809587,0.453444,-0.950567,-0.334453,-1.074104,-1.079946,0.388293,-0.449163,...,-1.131116,-0.396143,1.522375,-0.839238,-0.001083,-0.342914,0.385660,-0.000170,-3.385936,1.962038


## Saving to CSV
---

In [38]:
output_path = os.path.join(script_dir, '../../sim_env_data/v4_non_stat_zip_model_params.csv')
non_stat_zip_df.to_csv(output_path)